In [2]:
import sys
import os 
from pathlib import Path
import polars as pl
import duckdb
import pandas as pd

pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(50)
try:
    current_path = Path.cwd()
    if current_path.name == 'notebook' and current_path.parent.name == 'research':
        project_root = current_path.parent.parent
    else:
        project_root = current_path

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

    print(f" Project root terdeteksi: {project_root}")

except Exception as e:
    print(f" Gagal mendeteksi root. periksa struktur directory/folder/. Error: {e}")

 Project root terdeteksi: /home/bumip/orca


In [3]:
con = duckdb.connect(database=':memory:')
RAW_LAKE_PATH = project_root / "data" / "raw"
SILVER_LAKE_PATH = project_root / "data" / "silver"
print(f"Raw Lake Path   : {RAW_LAKE_PATH}")
print(f"Silver Lake Path: {SILVER_LAKE_PATH}")

Raw Lake Path   : /home/bumip/orca/data/raw
Silver Lake Path: /home/bumip/orca/data/silver


In [12]:
def inspect_lake_duckdb(lake_path: Path, lake_name: str):
    """
    Menggunakan DuckDB untuk melihat skema kolom DAN membaca internal metadata Parquet.
    """
    print(f"\n{'='*70}")
    print(f"DUCKDB INSPEKSI: {lake_name.upper()}")
    print(f"{'='*70}")
    
    if not lake_path.exists():
        print(f" ERROR: Path tidak ditemukan -> {lake_path}")
        return
        
    glob_path = str(lake_path / "**" / "*.parquet")
    
    try:
        # 1. CEK SKEMA BERSIH (Termasuk Hive Partitions)
        print("\n[1] SKEMA TABEL (HIVE AWARE):")
        schema_query = f"""
            DESCRIBE SELECT * FROM read_parquet('{glob_path}', hive_partitioning=1)
        """
        schema_df = con.execute(schema_query).df()
        for _, row in schema_df.iterrows():
            print(f" - {row['column_name']:<20} : {row['column_type']}")
            
        # 2. CEK INTERNAL METADATA (Menggunakan Kolom DuckDB Versi Terbaru)
        print("\n[2] PARQUET INTERNAL METADATA (Statistik Row Group):")
        # Kita gunakan DISTINCT karena metadata DuckDB memecah info per kolom
        meta_query = f"""
            SELECT DISTINCT
                file_name, 
                row_group_id, 
                row_group_num_rows,
                row_group_bytes
            FROM parquet_metadata('{glob_path}')
            LIMIT 5
        """
        meta_df = con.execute(meta_query).df()
        
        # Bersihkan nama file yang panjang agar rapi di terminal
        meta_df['file_name'] = meta_df['file_name'].apply(lambda x: Path(x).name)
        
        # Format bytes ke Megabytes agar mudah dibaca
        if 'row_group_bytes' in meta_df.columns:
            meta_df['size_mb'] = (meta_df['row_group_bytes'] / (1024 * 1024)).round(2)
            meta_df = meta_df.drop(columns=['row_group_bytes'])
            
        print(meta_df.to_string(index=False))
        print(" (Menampilkan max 5 row-group pertama...)")
        
    except Exception as e:
        print(f" Gagal memindai {lake_name} dengan DuckDB: {e}")

# --- CARA PENGGUNAAN ---
inspect_lake_duckdb(RAW_LAKE_PATH, "RAW DATA LAKE")
inspect_lake_duckdb(SILVER_LAKE_PATH, "SILVER DATA LAKE")


DUCKDB INSPEKSI: RAW DATA LAKE

[1] SKEMA TABEL (HIVE AWARE):
 - timestamp            : BIGINT
 - open                 : DOUBLE
 - high                 : DOUBLE
 - low                  : DOUBLE
 - close                : DOUBLE
 - volume               : DOUBLE
 - interval             : VARCHAR
 - month                : VARCHAR
 - symbol               : VARCHAR
 - year                 : BIGINT

[2] PARQUET INTERNAL METADATA (Statistik Row Group):
   file_name  row_group_id  row_group_num_rows  size_mb
data.parquet             0               44560     2.41
data.parquet             0               44640     2.25
data.parquet             0               40320     1.03
data.parquet             0               44560     1.13
data.parquet             0               43200     1.12
 (Menampilkan max 5 row-group pertama...)

DUCKDB INSPEKSI: SILVER DATA LAKE

[1] SKEMA TABEL (HIVE AWARE):
 - timestamp            : TIMESTAMP
 - close_BTC            : DOUBLE
 - close_DOGE           : DOUBLE
 - l

In [8]:
def inspect_lake_duckdb_full(lake_path: Path, lake_name: str):
    """
    DuckDB Lake Inspector - Mode Pembongkaran Total (Semua Metadata)
    """
    print(f"\n{'='*70}")
    print(f"📊 DUCKDB FULL INSPEKSI: {lake_name.upper()}")
    print(f"{'='*70}")
    
    if not lake_path.exists():
        print(f"❌ ERROR: Path tidak ditemukan -> {lake_path}")
        return
        
    glob_path = str(lake_path / "**" / "*.parquet")
    
    try:
        # 1. CEK SKEMA TABEL UTAMA
        print("\n[1] SKEMA TABEL DATA (HIVE AWARE):")
        schema_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{glob_path}', hive_partitioning=1)").df()
        for _, row in schema_df.iterrows():
            print(f" 🔹 {row['column_name']:<20} : {row['column_type']}")
            
        # 2. INTIP SEMUA KOLOM METADATA DUCKDB
        print("\n[2] DAFTAR KOLOM PARQUET METADATA YANG TERSEDIA:")
        meta_schema = con.execute(f"DESCRIBE SELECT * FROM parquet_metadata('{glob_path}')").df()
        # Gabungkan nama kolom jadi satu baris agar tidak menuhi layar
        all_meta_columns = meta_schema['column_name'].tolist()
        print(" | ".join(all_meta_columns))
        
        # 3. TAMPILKAN SEMUA METADATA (DENGAN PANDAS SCROLL)
        print("\n[3] SAMPLE METADATA (Menampilkan semua kolom):")
        meta_query = f"""
            SELECT *
            FROM parquet_metadata('{glob_path}')
            LIMIT 3
        """
        meta_df = con.execute(meta_query).df()
        
        # Bersihkan nama file agar rapi
        if 'file_name' in meta_df.columns:
            meta_df['file_name'] = meta_df['file_name'].apply(lambda x: Path(x).name)
            
        # Agar Pandas menampilkan semua kolom tanpa titik-titik (...)
        pd.set_option('display.max_columns', None)
        print(meta_df.head(3))
        pd.reset_option('display.max_columns')
        
    except Exception as e:
        print(f"❌ Gagal memindai {lake_name} dengan DuckDB: {e}")

# --- CARA PENGGUNAAN ---
inspect_lake_duckdb_full(RAW_LAKE_PATH, "RAW DATA LAKE")
# inspect_lake_duckdb_full(SILVER_LAKE_PATH, "SILVER DATA LAKE")


DUCKDB FULL INSPEKSI: RAW DATA LAKE

[1] SKEMA TABEL DATA (HIVE AWARE):
 - timestamp            : BIGINT
 - open                 : DOUBLE
 - high                 : DOUBLE
 - low                  : DOUBLE
 - close                : DOUBLE
 - volume               : DOUBLE
 - interval             : VARCHAR
 - month                : VARCHAR
 - symbol               : VARCHAR
 - year                 : BIGINT

[2] DAFTAR KOLOM PARQUET METADATA YANG TERSEDIA:
file_name | row_group_id | row_group_num_rows | row_group_num_columns | row_group_bytes | column_id | file_offset | num_values | path_in_schema | type | stats_min | stats_max | stats_null_count | stats_distinct_count | stats_min_value | stats_max_value | compression | encodings | index_page_offset | dictionary_page_offset | data_page_offset | total_compressed_size | total_uncompressed_size | key_value_metadata | bloom_filter_offset | bloom_filter_length | min_is_exact | max_is_exact | row_group_compressed_bytes | geo_bbox | geo_types

[3]

In [11]:
def inspect_lake_schema(lake_path: Path, lake_name: str):
    """
    Mengintip skema dari seluruh Data Lake tanpa memuat data ke RAM.
    Otomatis membaca Hive Partitions (misal: year=2024/month=01).
    """
    print(f"\n{'='*60}")
    print(f"INSPEKSI SKEMA: {lake_name.upper()}")
    print(f"{'='*60}")
    
    if not lake_path.exists():
        print(f"ERROR: Path tidak ditemukan -> {lake_path}")
        return
        
    try:
        # Gunakan Lazy Scan dengan hive_partitioning=True
        # Tanda **/*.parquet akan mencari semua file di dalam sub-folder bulan/tahun
        lazy_df = pl.scan_parquet(lake_path / "**" / "*.parquet", hive_partitioning=True)
        
        # Ekstrak Skema
        schema = lazy_df.collect_schema()
        
        print(f"Total Kolom: {len(schema)}")
        print("-" * 60)
        
        # Tampilkan nama kolom dan tipe datanya
        for col_name, dtype in schema.items():
            print(f" - {col_name:<25} : {dtype}")
            
    except Exception as e:
        print(f"Gagal memindai {lake_name}: {e}")

# --- CARA PENGGUNAAN ---
inspect_lake_schema(RAW_LAKE_PATH, "RAW DATA LAKE")
inspect_lake_schema(SILVER_LAKE_PATH, "SILVER DATA LAKE")


INSPEKSI SKEMA: RAW DATA LAKE
Total Kolom: 10
------------------------------------------------------------
 - timestamp                 : Int64
 - open                      : Float64
 - high                      : Float64
 - low                       : Float64
 - close                     : Float64
 - volume                    : Float64
 - symbol                    : String
 - interval                  : String
 - year                      : Int64
 - month                     : Int64

INSPEKSI SKEMA: SILVER DATA LAKE
Total Kolom: 21
------------------------------------------------------------
 - timestamp                 : Datetime(time_unit='ms', time_zone=None)
 - close_BTC                 : Float64
 - close_DOGE                : Float64
 - log_BTC                   : Float64
 - log_DOGE                  : Float64
 - ret_BTC                   : Float64
 - ret_DOGE                  : Float64
 - vol_BTC_1h                : Float64
 - vol_DOGE_1h               : Float64
 - corr_DOGE_BT